## 0. 당뇨병 데이터셋
1) load_diabetes() 데이터셋 설명
- 출처: scikit-learn에서 제공하는 내장 의료 데이터셋
- 용도: 당뇨병 진행도 예측(회귀)
- 샘플 수: 442명
- 특성 수: 10개 (모두 연속형, 이미 정규화된 상태)
- 타겟 값: 1년 후 환자의 당뇨병 진행 정도 (0~346 범위의 연속형 변수)

2) 데이터셋 컬럼 정보

| 컬럼명        | 설명         |
| ---------- | ---------- |
| `age`      | 나이 (표준화됨)  |
| `sex`      | 성별 (표준화됨)  |
| `bmi`      | 체질량 지수     |
| `bp`       | 평균 혈압      |
| `s1`\~`s6` | 혈청 검사 수치   |
| `target`   | 당뇨병 진행도 지표 |


# 0. 데이터 준비

## 0.1 라이브러리 설치

In [2]:
# !uv add  matplotlib  pandas seaborn scipy

## 0.2 라이브러리 import

In [3]:
import platform

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from scipy.stats import pearsonr, spearmanr, kendalltau

## 0.3 그래프 한글 설정

In [4]:
# OS에 따른 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우: 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')    # 맥: 애플 고딕
else:
    plt.rc('font', family='NanumBarunGothic') # 리눅스

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

# 1 데이터 생성  및 입출력

## 1.1 시리즈 

In [5]:
ages = [45, 56, 67, 70, 72]
s = pd.Series(ages)
print(s)
print("----------------------------------------------------------------------------")
s = pd.Series(ages, name='나이')
print(s)
print("----------------------------------------------------------------------------")
print(s.index)  # RangeIndex
print("----------------------------------------------------------------------------")
e = pd.Series(ages, index=['a','b','c','d','e'], name='나이')
print(e)
print("----------------------------------------------------------------------------")
print(e.index) # Index(['a', 'b', 'c', 'd', 'e'])

0    45
1    56
2    67
3    70
4    72
dtype: int64
----------------------------------------------------------------------------
0    45
1    56
2    67
3    70
4    72
Name: 나이, dtype: int64
----------------------------------------------------------------------------
RangeIndex(start=0, stop=5, step=1)
----------------------------------------------------------------------------
a    45
b    56
c    67
d    70
e    72
Name: 나이, dtype: int64
----------------------------------------------------------------------------
Index(['a', 'b', 'c', 'd', 'e'], dtype='str')


In [6]:
ages = [45, 56, 67, 70, 72]
m = pd.Series(ages)
print(dir(pd.Series))
print("----------------------------------------------------------------------------")
m.index = ['a', 'b','c','d','e']
print(m)
print("----------------------------------------------------------------------------")
m.name = '나이'
print(m)


['T', '_AXIS_LEN', '_AXIS_ORDERS', '_AXIS_TO_AXIS_NUMBER', '_HANDLED_TYPES', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_ufunc__', '__arrow_c_stream__', '__bool__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__finalize__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__imod__', '__imul__', '__init__', '__init_subclass__', '__invert__', '__ior__', '__ipow__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__or__', '__pandas_priority__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmatmul__', '__rmod__', '__rmul__', '

### 1.2 데이터프레임 

터미널이나 주피터 노트북 환경에서 데이터프레임 pandas.DataFrame을 출력할 때 컬럼이 너무 많거나 화면 너비를 넘어갈 경우 자동으로 줄바꿈되어 출력된다. 이걸 방지하고 모든 컬럼이 한 줄에 출력되도록 하려면 pandas의 출력 옵션을 조정하면 된다.

In [7]:
# 출력 너비 제한을 해제하여 여러 줄로 쪼개져(줄바꿈) 출력되는 현상 방지
pd.set_option('display.expand_frame_repr', False)

# 콘솔/터미널에 생략(...) 없이 출력할 최대 컬럼 수를 무제한(None)으로 설정
pd.set_option('display.max_columns', None)

In [8]:
data = {
    '환자ID': [101, 102, 103, 104],
    '나이': [65, 72, 58, 45],
    '진단결과': ['양성', '음성', '양성', '음성']
}

df = pd.DataFrame(data)
print(df)
print("----------------------------------------------------------------------------")
print(df.index) # RangeIndex(start=0, stop=4, step=1)
print("----------------------------------------------------------------------------")
print(df.columns) # Index(['환자ID', '나이', '진단결과'], dtype='object')
print("----------------------------------------------------------------------------")
print(df.shape) # (4, 3)
print("----------------------------------------------------------------------------")
df.to_csv('source/patients.csv', index=False)
df_loaded = pd.read_csv('source/patients.csv')
print('\n 저장 후 다시 불러오기:\n')
df_loaded

   환자ID  나이 진단결과
0   101  65   양성
1   102  72   음성
2   103  58   양성
3   104  45   음성
----------------------------------------------------------------------------
RangeIndex(start=0, stop=4, step=1)
----------------------------------------------------------------------------
Index(['환자ID', '나이', '진단결과'], dtype='str')
----------------------------------------------------------------------------
(4, 3)
----------------------------------------------------------------------------

 저장 후 다시 불러오기:



,환자ID,나이,진단결과
0,101,65,양성
1,102,72,음성
2,103,58,양성
3,104,45,음성


# 2. diabetes 데이터 로드

In [9]:
diabetes = load_diabetes()
print(dir(diabetes))
print("----------------------------------------------------------------------------")
# 컬럼명 출력
print(diabetes.feature_names)

['DESCR', 'data', 'data_filename', 'data_module', 'feature_names', 'frame', 'target', 'target_filename']
----------------------------------------------------------------------------
['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [10]:
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

In [11]:
print(df.shape)
print(df.target.shape)


(442, 11)
(442,)


In [12]:
print(dir(df))
print("----------------------------------------------------------------------------")
print(df.values)

['T', '_AXIS_LEN', '_AXIS_ORDERS', '_AXIS_TO_AXIS_NUMBER', '_HANDLED_TYPES', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_ufunc__', '__arrow_c_stream__', '__bool__', '__class__', '__contains__', '__copy__', '__dataframe__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__finalize__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__imod__', '__imul__', '__init__', '__init_subclass__', '__invert__', '__ior__', '__ipow__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__or__', '__pandas_priority__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmatmul__', '__rmod_

In [13]:
df

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0


In [14]:
df.iloc[:, :-1]

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930


In [15]:
# df.target
df["target"]

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [16]:
print(type(df))

<class 'pandas.DataFrame'>


# 3. 데이터 탐색(EDA)

In [17]:
# df = pd.DataFrame(df, columns= df.feature_names )
df['target'] = df['target']
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [18]:
# 상위데이터 5개 출력
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [19]:
df.head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0


In [20]:
# 아래 5개 출력
df.tail()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0
441,-0.045472,-0.044642,-0.073030,-0.081413,0.083740,0.027809,0.173816,-0.039493,-0.004222,0.003064,57.0


In [21]:
df.tail(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0
441,-0.045472,-0.044642,-0.073030,-0.081413,0.083740,0.027809,0.173816,-0.039493,-0.004222,0.003064,57.0


In [22]:
# random으로 1개 출력
df.sample()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
353,-0.052738,-0.044642,-0.055785,-0.036656,0.089244,-0.003193,0.008142,0.034309,0.132376,0.003064,109.0


In [23]:
df.sample(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
350,-0.027310,0.05068,0.060618,0.107944,0.012191,-0.017598,-0.002903,-0.002592,0.070207,0.135612,243.0
125,-0.005515,0.05068,-0.008362,-0.002228,-0.033216,-0.063630,-0.036038,-0.002592,0.080590,0.007207,161.0
227,0.067136,0.05068,-0.029918,0.057437,-0.000193,-0.015719,0.074412,-0.050564,-0.038460,0.007207,108.0


In [24]:
# info() 데이터 구조를 확인한다.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


describe(percentiles=None, include=None, exclude=None)
- percentiles	백분위수 지정 (예: [.1, .5, .9])
- include	포함할 자료형 지정 (예: include='all', include=['object'])
- exclude	제외할 자료형 지정 (예: exclude=['float'])

[옵션 설명]  
1. percentiles: list-like, default None
   출력할 백분위수(percentile) 를 지정합니다.

   기본값은 [0.25, 0.5, 0.75] (25%, 50%, 75%)입니다.

   예를 들어, **10%, 25%, 50%, 75%, 90%**를 출력하고 싶다면:  
   df.describe(percentiles=[.1, .25, .5, .75, .9])  
   > 백분위수는 0 < p < 1 사이의 값으로 지정해야 합다.  

2. include: ‘all’, list-like of dtypes or None, default None
   어떤 데이터 유형의 컬럼을 포함할지 지정한다.  

| 값                            | 의미                         |
| ---------------------------- | -------------------------- |
| `None`                       | 수치형(numeric) 데이터만 포함 (기본값) |
| `'all'`                      | 모든 컬럼 포함 (수치형, 문자열 등)      |
| `['object', 'float', 'int']` | 특정 자료형만 포함                 |

  df.describe(include='all')   # 모든 열에 대해 통계 제공
  df.describe(include=['object'])  # 문자열형 열만 포함

3. exclude: list-like of dtypes, default None
  어떤 데이터 유형의 컬럼을 제외할지를 지정한다.
  df.describe(exclude=['object'])  # 문자열(object) 제외
  df.describe(exclude=['float'])   # 실수형 제외

```python
import pandas as pd
import seaborn as sns

df = sns.load_dataset('titanic')
```

```python
# 1. 기본 describe
print(df.describe())

# 2. 모든 컬럼 포함
print(df.describe(include='all'))

# 3. 10%, 50%, 90% 백분위수 추가
print(df.describe(percentiles=[.1, .5, .9]))

# 4. object 타입 제외
print(df.describe(exclude=['object']))
```

In [25]:
# 기본 통계처리 함수
df.describe()  # describe(percentiles=None, include=None, exclude=None)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,442.000000
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.268604e-17,1.130318e-17,152.133484
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,77.093005
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01,25.000000
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02,87.000000
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03,140.500000
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02,211.500000
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01,346.000000


# 4.인덱싱 & 슬라이싱 실습

| 인덱서 | 대상 지정 방식 | 슬라이싱 미포함/포함 여부 | 비고 |
| --- | --- | --- | --- |
| **`[]`** | `df[열]`,`df[[열,열,열]]`  또는 `df[행 슬라이싱]` | 끝 번호 미포함 (`[:5]` -> 0~4) | 기본 탐색용 |
| **`df.loc[행, 열]`** | 행 라벨(이름), 열 라벨(이름) | **끝 라벨 포함** (`[:5]` -> 0~5) | 라벨/이름 기준 |
| **`df.iloc[행, 열]`** | 행 번호(위치), 열 번호(위치) | 끝 번호 미포함 (`[:5]` -> 0~4) | 순수 위치 정수 기준 |
| **`df.at[행, 열]`** | 단일 값 (행 라벨, 열 라벨) | 슬라이싱 불가 | 빠른 단일 접근 |
| **`df.iat[행, 열]`** | 단일 값 (행 번호, 열 번호) | 슬라이싱 불가 | 빠른 단일 접근 |

In [26]:
# print(df['bmi'].head())
#print(df[['sex', 'bmi', 'target']].head())
#print(df[df['bmi']>0.15])  # boolean indexing


# print(df[:5])  # 인덱스 이름이 아니라 0부터 시작하는 '위치(Position/Index Number)'
# print(df['bmi'][:5])  #df['열이름'][행번호]
# print(df[['bmi', 'bp']][:5])  # '열이름'에는 fancy indexing을 사용할 수 있고 행번호엔는 slicing을 사용할 수 있다. 단 반대는 안된다.

# print(df.loc[:5]) # 행이름
# print(df.loc[:5, 'bmi'])  # df.loc['행이름', '열이름']
# print(df.loc[:5, ['bmi', 'bp']])
# print(df.loc[[0,2,4], ['bmi', 'bp']]) # df.loc([,]) : 행, 열에 slicing, fancy indexing을 모두 사용할 수 있다.

# print(df.iloc[:5, 1:5])   # df.iloc[행번호, 열번호]

print(df.at[0,'bmi'])  # df.at['행이름','열이름']
print(df.iat[0, 2])    # df.iat[행번호, 열번호]

0.061696206518683294
0.061696206518683294


# 5. unique(), value_counts(), sort_values()

| 항목       | `unique()`        | `value_counts()`                  | `sort_values()`              |
| -------- | ----------------- | --------------------------------- | ---------------------------- |
| 목적    | 고유한 값 추출          | 값의 빈도수 계산                         | 값 기준 정렬                      |
|  적용 대상 | 시리즈 (`df['col']`) | 시리즈 (`df['col']`)                 | 시리즈 또는 데이터프레임                |
| 반환 형태 | `np.ndarray`      | `pd.Series` (index: 값, value: 빈도) | 정렬된 시리즈 또는 데이터프레임            |
|  주요 옵션 | 없음                | `normalize`, `sort`, `ascending`  | `by`, `ascending`, `inplace` |
|  사용 시기 | 고유 값 목록이 궁금할 때    | 분포/빈도 파악할 때                       | 정렬 기준으로 순서 정할 때              |

In [27]:
df['sex'].unique()

array([ 0.05068012, -0.04464164])

In [28]:
df['sex'].value_counts()

sex
-0.044642    235
 0.050680    207
Name: count, dtype: int64

In [29]:
df['bmi'].sort_values()  # default: ascending=True 오름차순

df['bmi'].sort_values(ascending=False) # 내림차순

367    0.170555
256    0.160855
366    0.137143
145    0.128521
262    0.127443
         ...   
247   -0.081653
10    -0.083808
358   -0.084886
381   -0.089197
281   -0.090275
Name: bmi, Length: 442, dtype: float64

# 6. isin() 함수
- 여러 값 중 포함 여부를 확인할 때 사용
- SQL의 IN (...)과 유사한 동작을 함
- df[column].isin([값1, 값2, 값3, ...])

In [30]:
df['bp'][0]

np.float64(0.0218723855140367)

In [31]:
df['bp'].isin([0.0218723855140367, 0.059744, -0.081413]).unique()
df[df['bp'].isin([0.0218723855140367, 0.059744, -0.081413])].head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
40,0.005383,0.050680,-0.008362,0.021872,0.054845,0.073215,-0.024993,0.034309,0.012551,0.094191,100.0


In [32]:
# 부정 조건 (~연산자)
df[~df['bp'].isin([0.0218723855140367, 0.059744, -0.081413])].head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0


# 7. query() 함수
- 문자열 형태로 조건식을 작성해서 간결하고 가독성 좋게 필터링하는 방법
- 일반적인 df[...] 조건식보다 직관적
- df.query("조건식")


In [33]:
# &대신에 and을 사용한다.
df.query('bmi > 0.05 and bp > 0').head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.05068,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
23,0.045341,0.05068,0.060618,0.031065,0.028702,-0.047347,-0.054446,0.071210,0.133597,0.135612,245.0
32,0.034443,0.05068,0.125287,0.028758,-0.053855,-0.012900,-0.102307,0.108111,0.000272,0.027917,341.0


In [34]:
# |대신에 or을 사용
df.query('s1 < 0 or s2 > 0.05').head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0


In [35]:
df['sex'][0]

np.float64(0.05068011873981862)

In [36]:
# not : ~대신에서 query에서는 not()을 사용한다.
df.query('not(sex == 0.05068011873981862)').head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [37]:
#변수 사입
threshold = 0.05
df.query('bmi> @threshold').head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.05068,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
8,0.041708,0.05068,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0
23,0.045341,0.05068,0.060618,0.031065,0.028702,-0.047347,-0.054446,0.071210,0.133597,0.135612,245.0


In [38]:
df['bp'][50]

np.float64(0.014986683562338177)

In [39]:
# isin과 query()
allowed = [0.0218723855140367, -0.019441826196154435, 0.014986683562338177]
df.query('bp in @allowed and target >= 100').head(3)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
30,-0.060003,-0.044642,0.044451,-0.019442,-0.009825,-0.007577,0.022869,-0.039493,-0.027129,-0.009362,129.0


# 8. 기본 통계 함수

| 함수            | 설명        | 반환값        | 예시                  |
| ------------- | --------- | ---------- | ------------------- |
| `count()`     | 결측치 제외 개수 | 시리즈        | `df.count()`        |
| `mean()`      | 평균        | 시리즈        | `df.mean()`         |
| `median()`    | 중앙값       | 시리즈        | `df.median()`       |
| `mode()`      | 최빈값       | 시리즈/데이터프레임 | `df.mode()`         |
| `std()`       | 표준편차      | 시리즈        | `df.std()`          |
| `var()`       | 분산        | 시리즈        | `df.var()`          |
| `min()`       | 최소값       | 시리즈        | `df.min()`          |
| `max()`       | 최대값       | 시리즈        | `df.max()`          |
| `quantile(q)` | 분위수       | 시리즈        | `df.quantile(0.25)` |
| `sum()`       | 합계        | 시리즈        | `df.sum()`          |
| `describe()`  | 요약통계량     | 데이터프레임     | `df.describe()`     |
| `corr()`      | 상관계수      | 데이터프레임     | `df.corr()`         |
| `cov()`       | 공분산       | 데이터프레임     | `df.cov()`          |


In [40]:
# 1. 결측치 제외한 값 개수
print("1. count()")
print(df.count())
print("----------------------------------------------------------------")
# 2. 평균
print("\n2. mean()")
print(df.mean())
print("----------------------------------------------------------------")
# 3. 중앙값
print("\n3. median()")
print(df.median())
print("----------------------------------------------------------------")
# 4. 최빈값
print("\n4. mode()")
print(df.mode().head(1))
print("----------------------------------------------------------------")
# 5. 표준편차
print("\n5. std()")
print(df.std())
print("----------------------------------------------------------------")
# 6. 분산
print("\n6. var()")
print(df.var())
print("----------------------------------------------------------------")
# 7. 최소값
print("\n7. min()")
print(df.min())
print("----------------------------------------------------------------")
# 8. 최대값
print("\n8. max()")
print(df.max())
print("----------------------------------------------------------------")
# 9. 분위수 (25%, 50%, 75%)
print("\n9. quantile()")
print(df.quantile([0.25, 0.5, 0.75]))
print("----------------------------------------------------------------")
# 10. 전체 합계
print("\n10. sum()")
print(df.sum())
print("----------------------------------------------------------------")
# 11. 상관계수
print("\n12. corr()")
print(df.corr())
print("----------------------------------------------------------------")
# 12. 공분산
print("\n13. cov()")
print(df.cov())


1. count()
age       442
sex       442
bmi       442
bp        442
s1        442
s2        442
s3        442
s4        442
s5        442
s6        442
target    442
dtype: int64
----------------------------------------------------------------

2. mean()
age      -2.511817e-19
sex       1.230790e-17
bmi      -2.245564e-16
bp       -4.797570e-17
s1       -1.381499e-17
s2        3.918434e-17
s3       -5.777179e-18
s4       -9.042540e-18
s5        9.268604e-17
s6        1.130318e-17
target    1.521335e+02
dtype: float64
----------------------------------------------------------------

3. median()
age         0.005383
sex        -0.044642
bmi        -0.007284
bp         -0.005670
s1         -0.004321
s2         -0.003819
s3         -0.006584
s4         -0.002592
s5         -0.001947
s6         -0.001078
target    140.500000
dtype: float64
----------------------------------------------------------------

4. mode()
        age       sex       bmi        bp        s1        s2        s3       

In [41]:
# 이상치 감지 (IQR)
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
iqr_mask = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR)))
print(iqr_mask)
print(iqr_mask.sum())

print('- IQR 방식 이상치 개수:', iqr_mask.sum().sum())

       age    sex    bmi     bp     s1     s2     s3     s4     s5     s6  target
0    False  False  False  False  False  False  False  False  False  False   False
1    False  False  False  False  False  False  False  False  False  False   False
2    False  False  False  False  False  False  False  False  False  False   False
3    False  False  False  False  False  False  False  False  False  False   False
4    False  False  False  False  False  False  False  False  False  False   False
..     ...    ...    ...    ...    ...    ...    ...    ...    ...    ...     ...
437  False  False  False  False  False  False  False  False  False  False   False
438  False  False  False  False  False  False  False  False  False  False   False
439  False  False  False  False  False  False  False  False  False  False   False
440  False  False  False  False  False  False  False  False  False  False   False
441  False  False  False  False  False  False   True  False  False  False   False

[442 rows x 11 

상관관계에서 method 옵션
- pearson, kendall, spearman

1. Pearson 상관계수 (피어슨 상관계수)  
- 측정 방식: 선형 상관관계(linear correlation) 측정
- 전제 조건:
  - 변수는 **연속형(수치형)**이어야 함  
  - 정규분포를 가정 (또는 거의 정규분포일 때 유리)  
  - **선형 관계(linearity)**가 있어야 의미 있음  
- 값의 범위: -1 ~ +1  
- 해석:  
  - +1: 완전한 양의 선형 관계
  - 0: 선형 관계 없음
  - -1: 완전한 음의 선형 관계
- 민감도: 이상치(outlier)에 매우 민감

2. Spearman 상관계수 (스피어만 순위 상관계수)
- 측정 방식: 순위 기반(rank-based) 상관관계
- 전제 조건:
  - 비선형 관계도 어느 정도 파악 가능 (단, 순차적 관계일 경우)
  - 데이터가 **연속형 or 서열형(ordinal)**이면 사용 가능
- 값의 범위: -1 ~ +1
- 해석:
  - 피어슨과 동일하게 해석 가능
  - 단, 순위 기반이므로 단조(monotonic) 관계 파악에 적합
- 민감도: 이상치에 덜 민감

3. Kendall's Tau (켄달의 타우)
- 측정 방식: 쌍(pairwise) 비교를 통한 순위 일관성 측정
- 전제 조건:
  - 순서형 데이터, 또는 순위를 줄 수 있는 데이터
  - 비선형적이고 단조적인 관계에 적합
- 값의 범위: -1 ~ +1
- 해석:
  - 두 변수의 순서쌍 일치 비율과 불일치 비율을 바탕으로 상관 측정
  - 해석은 스피어만과 유사하나 더 보수적인 경향
- 민감도: 이상치에 가장 덜 민감

※ 비교

| 항목     | Pearson   | Spearman          | Kendall's Tau      |
| ------ | --------- | ----------------- | ------------------ |
| 데이터 조건 | 연속형, 정규분포 | 연속형 또는 서열형        | 서열형 또는 순위 가능       |
| 관계 유형  | **선형**    | **단조(monotonic)** | **단조(monotonic)**  |
| 이상치 영향 | 매우 큼      | 중간                | 적음                 |
| 계산 방식  | 값 자체      | **순위**를 기반        | 순서쌍의 **일치/불일치** 비교 |
| 계산 복잡도 | 낮음        | 낮음                | 비교적 높음             |
| 해석 직관성 | 높음        | 높음                | 중간 (다소 보수적)        |

※어떤 경우에 무엇을 쓰면 좋을까?
- 데이터가 정규분포 + 선형 관계 기대: → Pearson
- 비선형이지만 단조적 관계 또는 이상치 존재 가능성 있음: → Spearman
- 데이터가 순위 데이터거나 이상치가 많고 보수적으로 해석하고 싶을 때: → Kendall's Tau


In [42]:
!uv pip list | grep scipy
# !uv pip list | findstr scipy

scipy                   1.18.0


In [51]:

from scipy.stats import pearsonr, spearmanr, kendalltau

# 샘플 데이터 생성
data = {
    'x': [10, 20, 30, 40, 50],
    'y_linear': [15, 25, 35, 45, 55],      # x와 선형 관계
    'y_monotonic': [1, 2, 3, 6, 10],       # 단조 증가 (비선형)
    'y_noisy': [10, 22, 29, 41, 100]       # 이상치 포함
}

df = pd.DataFrame(data)

# Pearson 상관계수
pearson_corr, _ = pearsonr(df['x'], df['y_linear'])
print("Pearson 상관계수:", round(pearson_corr, 3))

# Spearman 상관계수
spearman_corr, _ = spearmanr(df['x'], df['y_monotonic'])
print("Spearman 상관계수:", round(spearman_corr, 3))

# Kendall 상관계수
kendall_corr, _ = kendalltau(df['x'], df['y_noisy'])
print("Kendall 상관계수:", round(kendall_corr, 3))


Pearson 상관계수: 1.0
Spearman 상관계수: 1.0
Kendall 상관계수: 1.0


y_linear는 x와 완전한 선형 관계 → Pearson = 1  
y_monotonic은 비선형이지만 증가하는 관계 → Spearman = 1
y_noisy는 이상치가 포함되어 있을때 관계 → Kendall = 1  

In [44]:
df.plot